# ML Stock Ranking Prototype

Goal: build a first supervised ranking/classification target using the existing feature table and cached price data.

Suggested next step after this notebook works well: move feature/label preparation into `ai_models/alpha_model.py`.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

WindowsPath('c:/Users/bhuiyana/Documents/GitHub/ai-investment-lab')

In [2]:
import pandas as pd

from ai_models.feature_builder import build_feature_table

feature_result = build_feature_table(
    fundamentals_path="data/fundamentals_cache.parquet",
    prices_path="data/prices_cache.parquet",
    treasury_path="data/treasury_yields_cache.parquet",
    benchmark_ticker="SPY",
)
features = feature_result.features.reset_index()
features.head()

,Ticker,Momentum_12M,Volatility_63D,Drawdown_252D,Return_21D,Revenue_Growth_YoY_Pct,EBITDA_Margin,ROE,FreeCashFlow_Margin,Benchmark_Volatility,Benchmark_Trend,YC_10Y_2Y,YC_10Y_3M,YC_Inversion,YC_Volatility
0,A,-0.094352,0.256801,-0.295212,-0.128762,6.728111,0.27898,0.19946,NaN,0.119467,0.183914,0.52,0.48,0.0,0.031328
1,AAPL,0.170154,0.236271,-0.133331,-0.062606,6.425512,0.35100,1.52021,NaN,0.119467,0.183914,0.52,0.48,0.0,0.031328
2,ABBV,-0.010978,0.271282,-0.149803,-0.105293,8.566763,0.47832,62.25000,NaN,0.119467,0.183914,0.52,0.48,0.0,0.031328
3,ABNB,0.038696,0.332178,-0.089899,0.041120,10.259413,0.21028,0.30233,NaN,0.119467,0.183914,0.52,0.48,0.0,0.031328
4,ABT,-0.131591,0.272315,-0.209340,-0.049695,5.668653,0.27145,0.12961,NaN,0.119467,0.183914,0.52,0.48,0.0,0.031328


In [3]:
prices = pd.read_parquet(ROOT / "data" / "prices_cache.parquet")
prices["Ticker"] = prices["Ticker"].astype(str).str.upper().str.strip()
prices["Date"] = pd.to_datetime(prices["Date"], errors="coerce")
prices["AdjClose"] = pd.to_numeric(prices["AdjClose"], errors="coerce")
prices = prices.dropna(subset=["Ticker", "Date", "AdjClose"])
prices = prices.sort_values(["Ticker", "Date"])
prices.head()

,Ticker,Date,AdjClose,Close,Volume
0,A,2021-03-19,118.573891,122.690002,2597000
1,A,2021-03-22,119.463005,123.610001,1772900
2,A,2021-03-23,117.887703,121.980003,1338300
3,A,2021-03-24,116.785942,120.839996,1477500
4,A,2021-03-25,117.810371,121.900002,967300


In [4]:
# Prototype label: future 63-business-day return by ticker.
future_returns = []
for ticker, g in prices.groupby("Ticker"):
    s = g.set_index("Date")["AdjClose"].sort_index()
    fwd_63 = s.shift(-63) / s - 1.0
    if not fwd_63.dropna().empty:
        future_returns.append({"Ticker": ticker, "ForwardReturn63D": float(fwd_63.dropna().iloc[-1])})

labels = pd.DataFrame(future_returns)
dataset = features.merge(labels, on="Ticker", how="inner")
dataset.head()

,Ticker,Momentum_12M,Volatility_63D,Drawdown_252D,Return_21D,Revenue_Growth_YoY_Pct,EBITDA_Margin,ROE,FreeCashFlow_Margin,Benchmark_Volatility,Benchmark_Trend,YC_10Y_2Y,YC_10Y_3M,YC_Inversion,YC_Volatility,ForwardReturn63D
0,A,-0.094352,0.256801,-0.295212,-0.128762,6.728111,0.27898,0.19946,NaN,0.119467,0.183914,0.52,0.48,0.0,0.031328,-0.204019
1,AAPL,0.170154,0.236271,-0.133331,-0.062606,6.425512,0.35100,1.52021,NaN,0.119467,0.183914,0.52,0.48,0.0,0.031328,-0.096785
2,ABBV,-0.010978,0.271282,-0.149803,-0.105293,8.566763,0.47832,62.25000,NaN,0.119467,0.183914,0.52,0.48,0.0,0.031328,-0.077731
3,ABNB,0.038696,0.332178,-0.089899,0.041120,10.259413,0.21028,0.30233,NaN,0.119467,0.183914,0.52,0.48,0.0,0.031328,-0.019923
4,ABT,-0.131591,0.272315,-0.209340,-0.049695,5.668653,0.27145,0.12961,NaN,0.119467,0.183914,0.52,0.48,0.0,0.031328,-0.147049


In [6]:
# Optional if scikit-learn is installed.
try:
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import mean_absolute_error, r2_score

    numeric = dataset.select_dtypes(include=["number"]).drop(columns=["ForwardReturn63D"])
    y = dataset["ForwardReturn63D"]
    X = numeric.fillna(numeric.median(numeric_only=True))

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
    model = RandomForestRegressor(n_estimators=200, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print({"mae": mean_absolute_error(y_test, pred), "r2": r2_score(y_test, pred)})
except ModuleNotFoundError:
    print("Install scikit-learn to run the first ML ranking prototype.")

Install scikit-learn to run the first ML ranking prototype.
